In [1]:
# --- Environment setup: runs on Colab, does nothing locally (use the .venv kernel there) ---
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = "https://github.com/aaronfainman/BLO-Code.git"
    PROJECT = "/content/BLO-Code/TextClassification"
    if not os.path.isdir(PROJECT):
        # Private repo: add a GitHub token as a Colab secret named GH_TOKEN (key icon, left sidebar)
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
        !git clone -q https://{token}@github.com/aaronfainman/BLO-Code.git /content/BLO-Code
    %cd {PROJECT}
    !git pull -q
    # Installs what pyproject.toml lists into Colab's Python. torch is already satisfied
    # by Colab's CUDA build, so it is NOT reinstalled.
    !uv --version || pip install -q uv
    !uv pip install --system -q -r pyproject.toml

    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    
else:
    from dotenv import load_dotenv
    load_dotenv(override=True)



In [2]:

import numpy as np
import torch
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from itertools import islice



# Dataset utils

def stream(split, n, seed):
    # stream dataset rather than download full
    s = load_dataset("Yelp/yelp_review_full", split=split, streaming=True)
    return list(islice(s.shuffle(seed=seed, buffer_size=1000), n))

def trunc(text, n):
    return tok.decode(tok(text, add_special_tokens=False).input_ids[:n])


def fmt(text, n, answer=None):
    s = f"Review: {trunc(text, n)}\nRating (1-5 stars):"
    return s if answer is None else s + f" {answer}\n\n"


@torch.no_grad()
def score(prompts):
    """Return an array [len(prompts), 5] of log P(answer | prompt), summed over answer tokens."""
    seqs, cont_lens = [], []
    for p in prompts:
        p_ids = tok(p).input_ids
        for c_ids in VERB_IDS:
            seqs.append(p_ids + c_ids)
            cont_lens.append(len(c_ids))
    L = max(map(len, seqs))
    ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long)
    att = torch.zeros((len(seqs), L), dtype=torch.long)
    for k, s in enumerate(seqs):              # left padding
        ids[k, L - len(s):] = torch.tensor(s)
        att[k, L - len(s):] = 1
    pos = (att.cumsum(-1) - 1).clamp(min=0)
    ids, att, pos = ids.to(device), att.to(device), pos.to(device)

    # Run the transformer, then apply the LM head only at the positions we need.
    h = model.model(input_ids=ids, attention_mask=att, position_ids=pos).last_hidden_state
    out = []
    for k, cl in enumerate(cont_lens):
        hk = h[k, L - cl - 1 : L - 1]                       # positions predicting the answer tokens
        logp = model.lm_head(hk).float().log_softmax(-1)
        tgt = ids[k, L - cl :]
        out.append(logp.gather(-1, tgt[:, None]).sum().item())
    return np.array(out).reshape(len(prompts), 5)


def normalise(lp):
    """Log-softmax over the 5 candidate answers."""
    lp = lp - lp.max(-1, keepdims=True)
    return lp - np.log(np.exp(lp).sum(-1, keepdims=True))


# Model utils

MODEL = "HuggingFaceTB/SmolLM-135M"
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | device {device}")
dtype = torch.float16 if device == "cuda" else torch.float32

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype).to(device).eval()




/Users/acf22/Library/Mobile Documents/com~apple~CloudDocs/Documents/1. PhD/Articles+Presentations/8. ICLR 27/TextClassification/.venv.nosync/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python 3.12.8 | torch 2.15.0 | device mps


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W1118 13:53:56.775000 51215 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 5728.14it/s]


In [3]:



N_EVAL = 100          # test reviews to evaluate (the full test split is 50k)
K_SHOT = 1             # demonstrations per class in the prompt; 0 = zero-shot
MAX_REVIEW_TOK = 256   # truncation length for the query review
MAX_DEMO_TOK = 64      # truncation length for each demonstration review
BATCH = 8              # reviews per forward pass (x5 sequences each)
SEED = 0


test = Dataset.from_list(stream("test", N_EVAL, SEED))   # small in-memory Dataset; rest of script unchanged
# Verbs are candidate answers
VERBS = [f" {k}" for k in range(1, 6)]           # answers appended after the prompt
VERB_IDS = [tok(v, add_special_tokens=False).input_ids for v in VERBS]



# Few-shot prefix: K_SHOT examples per class, interleaved by class, from the train split.
prefix = ""
if K_SHOT > 0:
    pool = stream("train", 500, SEED + 1)
    per_class = [[ex for ex in pool if ex["label"] == c][:K_SHOT] for c in range(5)]
    demos = [per_class[c][k] for k in range(K_SHOT) for c in range(5)]
    rng = np.random.default_rng(SEED)
    rng.shuffle(demos)
    prefix = "".join(fmt(ex["text"], MAX_DEMO_TOK, ex["label"] + 1) for ex in demos)





In [7]:

# Content-free prompt for contextual calibration.
cf = normalise(score([prefix + fmt("N/A", MAX_REVIEW_TOK)]))[0]

all_lp, labels = [], []
for i in range(0, N_EVAL, BATCH):
    batch = test[i : i + BATCH]
    prompts = [prefix + fmt(t, MAX_REVIEW_TOK) for t in batch["text"]]
    all_lp.append(normalise(score(prompts)))
    labels += batch["label"]
    if (i // BATCH) % 25 == 0:
        print(f"{i + len(prompts)}/{N_EVAL}")

lp = np.concatenate(all_lp)
y = np.array(labels)


def report(name, pred):
    acc = (pred == y).mean()
    mae = np.abs(pred - y).mean()
    hist = np.bincount(pred, minlength=5) / len(pred)
    cm = np.zeros((5, 5), dtype=int)
    np.add.at(cm, (y, pred), 1)
    print(f"\n== {name} ==")
    print(f"accuracy {acc:.3f}   (chance 0.200)   MAE {mae:.3f} stars")
    print("predicted-star distribution:", np.round(hist, 3))
    print("confusion matrix (rows = true stars 1-5, cols = predicted):")
    print(cm)


report(f"{MODEL}, {K_SHOT}-shot per class, uncalibrated", lp.argmax(-1))
report(f"{MODEL}, {K_SHOT}-shot per class, contextually calibrated", (lp - cf).argmax(-1))

#5min48s 

8/100

== HuggingFaceTB/SmolLM-135M, 1-shot per class, uncalibrated ==
accuracy 0.260   (chance 0.200)   MAE 1.590 stars
predicted-star distribution: [0.84 0.03 0.13 0.   0.  ]
confusion matrix (rows = true stars 1-5, cols = predicted):
[[22  1  1  0  0]
 [19  0  5  0  0]
 [15  2  4  0  0]
 [17  0  1  0  0]
 [11  0  2  0  0]]

== HuggingFaceTB/SmolLM-135M, 1-shot per class, contextually calibrated ==
accuracy 0.200   (chance 0.200)   MAE 1.220 stars
predicted-star distribution: [0.   0.   0.92 0.08 0.  ]
confusion matrix (rows = true stars 1-5, cols = predicted):
[[ 0  0 19  5  0]
 [ 0  0 23  1  0]
 [ 0  0 20  1  0]
 [ 0  0 18  0  0]
 [ 0  0 12  1  0]]


# Bilevel finetuning

* **UL (θ)**: last decoder layer, trained with CE on the **validation** set.
* **LL (W)**: linear classification head, trained with CE on the **training** set.

Forward: frozen trunk (layers 0–28) → **last layer (θ)** → final RMSNorm (frozen) → pooling → **head (W)** → 5 logits.

The trunk is frozen, so its output (the hidden states entering the last layer) is computed **once** per dataset and cached; every loss evaluation then runs only one decoder layer.

### Data and trunk cache

In [ ]:
import importlib, time
import torch.nn.functional as F
import bilevel_model
importlib.reload(bilevel_model)   # pick up edits to bilevel_model.py without restarting the kernel
from bilevel_model import build_bilevel_models, encode_trunk, classify

N_TRAIN = 2000         # LL data: head trained on these   (Yelp train split)
N_VAL   = 500          # UL data: last layer trained on these (Yelp train split, disjoint from N_TRAIN)
N_TEST  = 1000         # held-out evaluation             (Yelp test split)
MAX_LEN = 256          # truncation length of the review, in tokens
POOLING = "last"       # "last" (the token after "Rating (1-5 stars):") or "mean"
DATA_SEED = 0

model.requires_grad_(False)

# Train and validation come from ONE stream of the train split, so they cannot overlap.
pool_ = stream("train", N_TRAIN + N_VAL, DATA_SEED)
splits = {"train": pool_[:N_TRAIN], "val": pool_[N_TRAIN:], "test": stream("test", N_TEST, DATA_SEED)}

# Cache the frozen trunk's output (hidden states entering the last layer) for every split.
cache = {}
for name, rows in splits.items():
    t0 = time.time()
    H, A = encode_trunk(model, tok, [fmt(r["text"], MAX_LEN) for r in rows],
                        batch_size=32, store_device=device)
    y = torch.tensor([r["label"] for r in rows], device=device)
    cache[name] = {"h": H, "attn": A, "y": y}
    print(f"{name:5s}: {len(rows):5d} examples, T={H.shape[1]}, "
          f"{H.numel() * H.element_size() / 2**30:.2f} GiB, "
          f"label counts {torch.bincount(y, minlength=5).tolist()}, {time.time() - t0:.0f}s")

### Model

In [ ]:
### Model: rep = UL model (theta), head = LL model (W)
# build_bilevel_models returns FRESH copies each call (pretrained last layer + newly
# initialised head), so call it once per run; `seed` fixes the head initialisation.
rep, head = build_bilevel_models(model, n_classes=5, pooling=POOLING, train_norm=True, seed=0)

def n_params(m, trainable=True):
    return sum(p.numel() for p in m.parameters() if p.requires_grad == trainable)

print(f"UL theta (last layer + final norm): {n_params(rep):,} trainable, {n_params(rep, False):,} frozen")
print(f"LL W (head):                        {n_params(head):,} trainable")

# Sanity check: shapes, and an untrained head should give CE close to log 5
with torch.no_grad():
    b = slice(0, 16)
    logits = classify(rep, head, cache["val"]["h"][b], cache["val"]["attn"][b])
    print("logits", tuple(logits.shape), "| CE", F.cross_entropy(logits, cache["val"]["y"][b]).item(),
          "(log 5 =", round(np.log(5), 3), ")")

UL theta (last layer + final norm): 3,540,672 trainable, 0 frozen
LL W (head):                        2,885 trainable
logits (16, 5) | CE 1.5569443702697754 (log 5 = 1.609 )


### Loss

In [ ]:
### Loss functions (BLO API: loss_fn(ll_model, data, ul_model, stochastic, seed, **kwargs) -> scalar)
# data = {"train": cache["train"], "val": cache["val"]}; each split is {"h", "attn", "y"}.
# LL (head W):           CE on a TRAIN minibatch.
# UL (representation θ): CE on a VAL minibatch.
#
# The optimiser's `seed` picks the minibatch, so equal seeds give identical batches. PABLO's four
# per-step seeds therefore give four independent samples; the other algorithms use one seed per step.
# With stochastic=False the seed is ignored and the full split is used (deterministic objective;
# fine for the head, but a full-split forward through the layer is memory-hungry).
BATCH_LL = 64   # train minibatch size (lower level)
BATCH_UL = 64   # val minibatch size (upper level)

def minibatch(split, stochastic, seed, batch_size):
    """Return (h, attn, y) for a seed-determined minibatch of a cached split (or the whole split)."""
    n = split["y"].shape[0]
    if not stochastic:
        return split["h"], split["attn"], split["y"]
    g = torch.Generator().manual_seed(int(seed))
    idx = torch.randperm(n, generator=g)[:batch_size].to(split["y"].device)   # without replacement
    return split["h"][idx], split["attn"][idx], split["y"][idx]


def lower_loss(ll_model, data, ul_model, stochastic=False, seed=None, **kwargs):
    """LL objective J(W; θ): cross-entropy of the head on training data."""
    h, attn, y = minibatch(data["train"], stochastic, seed, BATCH_LL)
    logits = ll_model(ul_model(h, attn))
    return F.cross_entropy(logits, y)


def upper_loss(ll_model, data, ul_model, stochastic=False, seed=None, **kwargs):
    """UL objective F(W, θ): cross-entropy of the head on validation data."""
    h, attn, y = minibatch(data["val"], stochastic, seed, BATCH_UL)
    logits = ll_model(ul_model(h, attn))
    return F.cross_entropy(logits, y)


data = {"train": cache["train"], "val": cache["val"]}

# Sanity check: same seed -> same batch/loss; different seed -> different batch; ≈ log 5 at init
with torch.no_grad():
    a, b, c = (lower_loss(head, data, rep, True, s).item() for s in (1, 1, 2))
    print(f"LL CE seed1={a:.4f} seed1={b:.4f} seed2={c:.4f} | UL CE seed1={upper_loss(head, data, rep, True, 1).item():.4f}")

### Methods and hyperparameters

In [ ]:
### Methods and hyperparameters
import importlib
from functools import partial
import blo_optimisers
importlib.reload(blo_optimisers)
from blo_optimisers import (BLO_optimiser_PABLO_Prox_Decouple_Bound, BLO_optimiser_BOME, BLO_optimiser_TSP,
                            BLO_optimiser_AID_Neumann, BLO_optimiser_AAL, BLO_optimiser_PBGD_Free,
                            BLO_optimiser_F2SA, BLO_optimiser_VPBGD)

SGD_MOMENTUM = partial(torch.optim.SGD, momentum=0.9)

# Methods to run. STABLE is not supported here: it materialises H_xy densely (3.5M x 2,885 entries,
# ~41 GB) and needs one backward pass per head parameter at every step.
METHODS = ["PABLO", "BOME", "AID", "F2SA", "VPBGD", "TSP", "AAL", "PBGD-Free"]

# STARTING POINTS ONLY - none of these are tuned for this problem yet.
HPARAMS = {
    "PABLO": dict(lr_w=1e-2, lr_theta=1e-4, lr_lambda=1e-2, rho=1e-2, alpha=0.99, eps=1e-8,
                  tau=0.1, y_rate=0.5, lambda_bound=1.0, grad_clip=10.0,
                  total_steps=None, warmup_ratio=0.1),   # total_steps="auto" -> warmup+cosine over num_iters
    "BOME":  dict(opt="sgd", inner_steps=10, inner_lr=1e-2, eta=0.5, lr_w=1e-2, lr_theta=1e-4,
                  barrier="grad_norm"),
    "F2SA":  dict(lr_w=1e-2, lr_x=1e-3, lambda_init=1.0, delta=0.01, inner_steps=1),
    "VPBGD": dict(lr_w=1e-2, lr_theta=1e-3, inner_lr=1e-2, gamma=1.0, inner_steps=5),
    "TSP":   dict(alpha=1e-3, beta=1e-2, eta=1e-2, gamma=0.1, delta=0.0, p=0.1, tau=1e-3,
                  mu=0.05, theta=0.05, omega=0.4, lambda_init=0.1),
    "AAL":   dict(rho0=0.1, alpha=1e-3, eps=1e-6, t=100.0, tau=1e-3),
    "PBGD-Free": dict(gamma=10.0, eta=1e-3, eta_gamma=1e-2, K=1),
}


def build_optimiser(method, rep, head, h, seed, num_iters):
    """BLO optimiser for (head = LL W, rep = UL theta) with hyperparameters h and run seed `seed`."""
    common = dict(ll_model=head, ul_model=rep, lower_loss_fn=lower_loss, upper_loss_fn=upper_loss,
                  stochastic=True, device=device, seed=seed)

    if method == "PABLO":
        rms = {"alpha": h["alpha"], "eps": h["eps"]}
        total_steps = num_iters if h.get("total_steps") == "auto" else h.get("total_steps")
        return BLO_optimiser_PABLO_Prox_Decouple_Bound(**common,
            w_optimiser=torch.optim.RMSprop, theta_optimiser=torch.optim.RMSprop,
            lambda_optimiser=torch.optim.SGD, w_optimiser_kwargs=rms, theta_optimiser_kwargs=rms,
            lr_w=h["lr_w"], lr_theta=h["lr_theta"], rho=h["rho"], lr_lambda=h["lr_lambda"],
            tau=h["tau"], gamma=h["y_rate"] / h["tau"], lambda_bound=h["lambda_bound"],
            grad_clip=h["grad_clip"], total_steps=total_steps, warmup_ratio=h.get("warmup_ratio", 0.1))

    if method == "BOME":
        h = dict(h)
        opt = {"sgdm": SGD_MOMENTUM, "adam": torch.optim.Adam, "rmsprop": torch.optim.RMSprop}[h.pop("opt", "sgdm")]
        return BLO_optimiser_BOME(**common, w_optimiser=opt, theta_optimiser=opt, **h)

    classes = {"AID": BLO_optimiser_AID_Neumann, "F2SA": BLO_optimiser_F2SA, "VPBGD": BLO_optimiser_VPBGD,
               "TSP": BLO_optimiser_TSP, "AAL": BLO_optimiser_AAL, "PBGD-Free": BLO_optimiser_PBGD_Free}
    if method in classes:
        return classes[method](**common, **h)
    raise ValueError(f"unknown method: {method}")

### Running an experiment
`run_experiment(method, seed, hparams, num_iters)` builds fresh models, trains, and returns the per-step history plus periodic train/val/test evaluations.

In [ ]:
### Running one experiment
import math

@torch.no_grad()
def evaluate(rep, head, split, batch_size=128):
    """Full-split CE and accuracy of the current (rep, head)."""
    was_training = rep.training, head.training
    rep.eval(); head.eval()
    n, loss, correct = split["y"].shape[0], 0.0, 0
    for i in range(0, n, batch_size):
        logits = classify(rep, head, split["h"][i:i + batch_size], split["attn"][i:i + batch_size])
        y = split["y"][i:i + batch_size]
        loss += F.cross_entropy(logits, y, reduction="sum").item()
        correct += (logits.argmax(-1) == y).sum().item()
    rep.train(was_training[0]); head.train(was_training[1])
    return {"loss": loss / n, "acc": correct / n}


def evaluate_all(rep, head, it):
    out = {"iter": it}
    for name in ("train", "val", "test"):
        for k, v in evaluate(rep, head, cache[name]).items():
            out[f"{name}_{k}"] = v
    return out


def run_experiment(method, seed, hparams, num_iters, eval_every=50, verbose=True):
    """One run of `method` from fresh models. `seed` sets the head init AND the optimiser's sample stream.

    Returns a dict:
      history : {metric: [value per step]} - whatever opt.step() reports (keys differ by method)
      evals   : [{"iter", "train_loss", "train_acc", "val_loss", "val_acc", "test_loss", "test_acc"}, ...]
                at iteration 0, every `eval_every` steps, and at the end
      final   : the last entry of evals
    """
    rep, head = build_bilevel_models(model, n_classes=5, pooling=POOLING, train_norm=True, seed=seed)
    opt = build_optimiser(method, rep, head, hparams, seed, num_iters)

    history, evals, diverged = {}, [evaluate_all(rep, head, 0)], False
    t0 = time.time()
    for it in range(1, num_iters + 1):
        stats = opt.step(data) or {}
        for k, v in stats.items():
            history.setdefault(k, []).append(float(v))
        if not all(math.isfinite(history[k][-1]) for k in ("ll_loss", "ul_loss") if k in history):
            diverged = True
            if verbose: print(f"  [{method} seed {seed}] diverged at iteration {it}")
            break
        if it % eval_every == 0 or it == num_iters:
            evals.append(evaluate_all(rep, head, it))
            if verbose:
                e = evals[-1]
                print(f"  [{method} seed {seed}] it {it:5d} | train CE {e['train_loss']:.3f} acc {e['train_acc']:.3f}"
                      f" | val CE {e['val_loss']:.3f} acc {e['val_acc']:.3f}"
                      f" | test acc {e['test_acc']:.3f} | {time.time() - t0:.0f}s")
    if diverged:
        evals.append(evaluate_all(rep, head, it))

    result = dict(method=method, seed=seed, hparams=dict(hparams), num_iters=num_iters, iters_run=it,
                  diverged=diverged, time=time.time() - t0, history=history, evals=evals, final=evals[-1])
    del opt, rep, head
    if device == "cuda": torch.cuda.empty_cache()
    return result

### Experiments

In [ ]:
### Experiments: every method x every seed
import os, pickle

N_ITERS    = 1000
SEEDS      = range(5)       # run seeds (head init + sample stream)
EVAL_EVERY = 50
SAVE_PATH  = None           # e.g. "results/blo_yelp.pkl" to pickle RESULTS after every run

RESULTS = {}                # RESULTS[method][seed] = run_experiment(...) output
for method in METHODS:
    RESULTS[method] = {}
    for seed in SEEDS:
        print(f"{method} | seed {seed}")
        RESULTS[method][seed] = run_experiment(method, seed, HPARAMS[method], N_ITERS, EVAL_EVERY)
        if SAVE_PATH:
            os.makedirs(os.path.dirname(SAVE_PATH) or ".", exist_ok=True)
            with open(SAVE_PATH, "wb") as f:
                pickle.dump(RESULTS, f)

# Summary: final test accuracy, mean ± std over seeds
for method, runs in RESULTS.items():
    acc = np.array([r["final"]["test_acc"] for r in runs.values()])
    n_div = sum(r["diverged"] for r in runs.values())
    print(f"{method:10s} test acc {acc.mean():.3f} ± {acc.std():.3f}  ({len(acc)} seeds, {n_div} diverged)")